DATABASE MANAGEMENT ASSIGNMENT
      ************

- Cleaning of ExpertReviews Data set.      

In [1]:
# Import Libraries: Pandas specifically for data, allows use of dataframes, cleaning, analysis. and
# Numpy specifically for calculations.


import pandas as pd
import numpy as np

df = pd.read_excel("ExpertReviewsClean43LIWC.xlsx")


In [ ]:

print("The size of the table:", df.shape)

print(df.info())

print("The columns names are: ", df.columns)

#dattypecolumn = df.columns

for i, column in enumerate(df.columns, start=1):
    print(f"{i}. {column}: {df[column].dtype}")
    
#print("The DateP column Datatype", df["dateP"].dtype)
print(df.dtypes.value_counts())
print(df.head())

In [ ]:
quality_report = pd.DataFrame({
    "column": df.columns,
    "data_type": df.dtypes.astype(str),
    "missing_values": df.isna().sum().values,
    "missing_percent": (
        df.isna().mean().values * 100
    ),
    "unique_values": df.nunique().values
})

print(quality_report)

APPLY THE MOJIBAKE FUNCTION TO CLEAN THE NAMES

In [4]:
#Definition of the function to clean the mojibake names
def fix_mojibake(raw):
    """ Mojibake is garbled unreadable text that happens when
    a computer miscodes a file. This cleans that up
    """
    if pd.isna(raw):
        return
    s = str(raw)
    try:
        return s.encode("latin1").decode("utf-8")
    except (UnicodeDecodeError, UnicodeEncodeError):
        return s 

In [5]:
#FIX MOJIBAKE NAMES/TEXT

df = df.reset_index(drop=True)
df["reviewer"] = df["reviewer"].apply(fix_mojibake)
df["reviewer_norm"] = df["reviewer"].astype(str).str.strip().str.lower()

"""This makes sure that everything is equal"""
pd.set_option("display.max_rows", None)
#print(df["reviewer_norm"])

#print(df.columns)

FOR THE INCONSISTENT Unknown or missing values, they are replaced by NaN!

In [ ]:
for column in df.columns:
    values = df[column].dropna().astype(str).str.strip()
    
    matches = values[values.isin(["None", "none", "NULL", "null", "N/A", "n/a", "nONE"])]

    
    if len(matches) > 0:
        print(f"\nColumn: {column}")
        print(matches.value_counts())
        



In [ ]:

#How many rows have UNKNOWN values in "dateP" and "reviewer" columns, but still have url

none_rows = df[df["dateP"].astype(str).str.strip().str.lower() == "none"]
none_reviewer_with_url =df[df["reviewer"].astype(str).str.strip().str.lower() == "none"]

count = none_rows["url"].notna().sum()
countrevurl = none_reviewer_with_url["url"].notna().sum()

print(count)
print (countrevurl)


#The results show that there are, so these fields are still used.


In [ ]:
#Search the entire dataframe for any unknown values, including none, null, and n/a

none_rows = df[
    df.astype(str)
      .apply(lambda col: col.str.strip().str.lower().isin(
          ["none", "null", "n/a"]
      ))
      .any(axis=1)
]
print(len(none_rows))
print(none_rows.isna().sum())

none_rows = none_rows.copy()

none_rows["missing_count"] = none_rows.isna().sum(axis=1)
none_rows["populated_count"] = none_rows.notna().sum(axis=1)

print(none_rows[["missing_count", "populated_count"]])

In [ ]:
#For all the unknown values replace them with NaN

df = df.replace(
    r"^\s*(None|none|NULL|null|N/A|n/a|nONE)\s*$",
    np.nan,
    regex=True
)
print(df.isna().sum())

In [ ]:
missing_reviewers = df[df["reviewer"].isnull()]

print(missing_reviewers)

df.isnull().sum()

In [ ]:
#print(df.dtypes) Just checking the datatypes
print(df.dtypes.value_counts())
print("The URL Datatype", df["url"].dtype)
print("The Date Datatype", df["dateP"].dtype)
print("The Reviwer Datatype", df["reviewer"].dtype)


Replacing the None/null/N/A data


In [12]:
for column in df.columns:
    values = df[column].dropna().astype(str).str.strip()
    
    matches = values[
        values.str.lower().isin(["none", "null", "n/a"])
    ]
    
    if len(matches) > 0:
        print(f"\nColumn: {column}")
        print(matches.value_counts())

NEXT STEP: CLEANING REVIEWER AND URL COLUMN. TO MAKE SURE THEY ARE STANDARDIZED.

In [ ]:
# Clean reviewer names
df["reviewer_norm"] = df["reviewer"].str.strip()
df["reviewer_norm"] = df["reviewer_norm"].str.replace(r"\s+", " ", regex=True)

# Clean URLs
df["url"] = df["url"].str.strip()
df["url"] = df["url"].str.replace(" ", "", regex=False)

print(df[["reviewer_norm", "url"]].head(20))
print(df[["reviewer_norm", "url"]].dtypes)

STANDARDIZE THE QUOTATIONS ON REVIEWER COLUMN

In [ ]:
#The single or double quotations are removed, ensuring that names like J., K. Are not affected.

df["reviewer_norm"] = (
    df["reviewer_norm"]
    .str.strip()
    .str.replace(r"""^['"]+|['"]+$""", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
)

print(df["reviewer"].dropna().head(30))

EXPORT THE DATA TO CSV

In [15]:
df.to_csv("ExpertReviewsCleanedF.csv", index=False)

QUALITY CHECK TO ENSURE THE FILE WAS REALLY CREATED

In [ ]:
df_check = pd.read_csv("ExpertReviewsCleanedF.csv")

print(df_check.shape)
print(df_check.head())
print(df_check.dtypes)
print(df["reviewer_norm"].head())

CHECK THE REVIEWERS NAMES

In [ ]:
#Creating a new Data Frame for Reviewers. With the intention to create a separate table in database.

reviewers = df_check[["reviewer_norm"]]
reviewers = reviewers.drop_duplicates()
reviewers = reviewers.reset_index(drop=True)

#Number of Unique Reviewers 
print("Number of reviewers:", len(reviewers))
print("Number of missing reviewers:", reviewers["reviewer_norm"].isna().sum())
reviewers.head()

#pd.set_option("display.max_rows", None)
#print(reviewers)

In [ ]:
#Moving the cleaned column to CSV

import pandas as pd

# 1. Clean and filter the reviewer column
clean_reviewers_series = (
    reviewers['reviewer_norm']
    .dropna()                             # Remove true NaNs / NULLs
    .astype(str)
    .str.replace(r'\s+', ' ', regex=True) # Replace hidden unicode/newlines/tabs with a single space
    .str.strip()                          # Remove leading/trailing spaces
)

# Invalid values to discard (all lowercase for case-insensitive matching)
invalid_values = {'nan', 'none', 'null', 'n/a', '', 'reviewer_norm'}

# Create mask using ~ (NOT) to KEEP values NOT in invalid_values
valid_mask = ~clean_reviewers_series.str.lower().isin(invalid_values)

# Get array of unique valid reviewer names
clean_reviewers = clean_reviewers_series[valid_mask].unique()

# 2. Convert to a single-column DataFrame
df_clean_reviewers = pd.DataFrame(clean_reviewers, columns=['reviewer_name'])

print(f"Total clean unique reviewers found: {len(df_clean_reviewers)}")
print(df_clean_reviewers.head())

# 3. Export the CLEANED DataFrame (df_clean_reviewers) to CSV
df_clean_reviewers.to_csv("cleaner_reviewers.csv", index=False, header=False)

In [ ]:
import os
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv

load_dotenv()

db_password = os.getenv("DB_PASSWORD")

Connection = psycopg2.connect(
    dbname='Database_Assignemtnt_DDB',
    user='postgres',
    password=db_password,
    host='localhost',
    port='5432'
)

cur = Connection.cursor()

# 1. Clear stuck transactions if any
Connection.rollback()

# 2. Wipe the table and reset IDs back to 1
cur.execute("TRUNCATE TABLE Reviewer RESTART IDENTITY CASCADE;")

# 3. Open the CSV file and bulk copy into the reviewer_name column
with open("cleaner_reviewers.csv", "r", encoding="utf-8") as f:
    COPY_SQL = "COPY Reviewer (reviewer_name) FROM STDIN WITH (FORMAT csv);"
    cur.copy_expert(COPY_SQL, f)

# 4. Commit changes to PostgreSQL
Connection.commit()

print("Successfully truncated table, reset IDs, and imported clean CSV!")




In [ ]:
# 5. Verify the top 10 rows in Python
df_check = pd.read_sql("SELECT * FROM Reviewer LIMIT 10;", Connection)
print(df_check)

GENRE, MOVIE GENRE Data CLEANING AND LOADING

In [ ]:
#Load the data from storage

meta_df = pd.read_excel("metaClean43Brightspace.xlsx")

In [ ]:

# -------------------------------------------------------------------
# 1. Extract, Split, and Explode Genres
# -------------------------------------------------------------------
# Metacritic dataframe is `meta_df` and column is 'genres'
raw_genres = (
    meta_df['genre']
    .dropna()                             # Drop NaNs
    .astype(str)                          # Convert to string
    .str.split(',')                       # Split comma-separated genres into lists
    .explode()                            # Flatten lists so each genre gets its own row
    .str.replace(r'\s+', ' ', regex=True) # Normalize inner whitespace/tabs
    .str.strip()                          # Strip leading/trailing spaces
)

# Unwanted values filter
invalid = {'', 'nan', 'none', 'null', 'n/a'}

# Keep unique valid genres
clean_genres = sorted([g for g in raw_genres.unique() if g.lower() not in invalid])

df_genres = pd.DataFrame(clean_genres, columns=['genre_name'])

print(f"Extracted {len(df_genres)} unique genres:")
print(df_genres.head(10))

df_genres.to_csv("clean_genres.csv", index=False, header=False)

In [37]:
#ADD THE GENRE RECORDS IN POSTGRES

# Clear any open locks
Connection.rollback()

with Connection.cursor() as cur:
    # 3. Truncate Genre table and reset genre_id back to 1
    cur.execute("TRUNCATE TABLE Genre RESTART IDENTITY CASCADE;")
    
    # 4. Stream clean CSV into PostgreSQL
    with open("clean_genres.csv", "r", encoding="utf-8") as f:
        COPY_SQL = "COPY genre (genre_name) FROM STDIN WITH (FORMAT csv);"
        cur.copy_expert(COPY_SQL, f)

# 5. Commit changes
Connection.commit()
print("Genre table successfully created and populated!")

# 6. Verify in Python
#df_check = pd.read_sql("SELECT * FROM Genre;", Connection)
#print(df_check)

#Connection.close()

Genre table successfully created and populated!


************/// **  MOVIE TABLE   ** // **************